In [ ]:
# =====================================================================
# Surrogate Model Analysis
#   Part B: Sensitivity Heatmap  (xc-zc scan at fixed Lx,Lz)
#   Part A: Ultra-Dense Pareto Front via NSGA-II
# =====================================================================

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# 1. Load Surrogate Model
# ============================================================
class ResidualMLP(nn.Module):
    def __init__(self, in_dim=4, hidden=256, n_layers=4, dropout=0.05):
        super().__init__()
        self.input_layer = nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.BatchNorm1d(hidden))
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(hidden,hidden),nn.ReLU(),nn.BatchNorm1d(hidden),nn.Dropout(dropout)) for _ in range(n_layers)])
        self.output_layer = nn.Linear(hidden,1)
    def forward(self,x):
        h=self.input_layer(x)
        for b in self.blocks: h=h+b(h)
        return torch.sigmoid(self.output_layer(h))

import os, glob
model_path = None
for c in ['surrogate_model.pt','/kaggle/working/surrogate_model.pt']+sorted(glob.glob('/kaggle/input/**/surrogate_model.pt',recursive=True)):
    if os.path.exists(c): model_path=c; break
if model_path is None: raise FileNotFoundError('surrogate_model.pt not found')
print(f'Loading from: {model_path}')
ckpt = torch.load(model_path, weights_only=False, map_location=device)
lb = ckpt['lb'].cpu().numpy()
ub = ckpt['ub'].cpu().numpy()
model = ResidualMLP().to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Model loaded: R²={ckpt["r2"]:.4f}, MAE={ckpt["mae"]*100:.2f}%')

def predict_outage(X_np):
    """X_np: [N,4] raw coords → predicted outage [N]"""
    X_norm = (X_np - lb) / (ub - lb)
    X_t = torch.tensor(X_norm, dtype=torch.float32, device=device)
    with torch.no_grad():
        return model(X_t).cpu().numpy().flatten()

# Warm-up
test = np.array([[5.0, 1.5, 0.15, 0.15]])
print(f'Warm-up: window[5,1.5,0.15,0.15] → outage={predict_outage(test)[0]*100:.2f}%\n')

# ============================================================
# PART B — Sensitivity Heatmap: xc−zc scan
# ============================================================
Lx_fixed, Lz_fixed = 0.15, 0.15
N_xc, N_zc = 500, 200

xc_vals = np.linspace(lb[0], ub[0], N_xc)
zc_vals = np.linspace(lb[1], ub[1], N_zc)
XC, ZC = np.meshgrid(xc_vals, zc_vals)  # [N_zc, N_xc]

# Batch-predict all grid points
grid_input = np.column_stack([XC.ravel(), ZC.ravel(),
                               np.full_like(XC.ravel(), Lx_fixed),
                               np.full_like(XC.ravel(), Lz_fixed)])
outage_map = predict_outage(grid_input).reshape(N_zc, N_xc)

# Find best xc,zc for this Lx,Lz
best_idx = np.argmin(outage_map)
best_z, best_x = np.unravel_index(best_idx, outage_map.shape)
print(f'Best (xc,zc) for Lx={Lx_fixed},Lz={Lz_fixed}: '
      f'({xc_vals[best_x]:.2f},{zc_vals[best_z]:.2f}) → outage={outage_map[best_z,best_x]*100:.2f}%')

# Plot
fig1, ax1 = plt.subplots(figsize=(10, 4))
im = ax1.pcolormesh(xc_vals, zc_vals, outage_map*100, cmap='RdYlGn_r', shading='auto', vmin=0, vmax=50)
ax1.plot(xc_vals[best_x], zc_vals[best_z], 'b*', markersize=14, markeredgewidth=1.5)
ax1.axhline(y=1.5, color='gray', linestyle='--', alpha=0.4)
cb = plt.colorbar(im, ax=ax1, label='Predicted Outage [%]')
cb.ax.axhline(y=10, color='red', linestyle='--', linewidth=1.5)
ax1.set_xlabel('xc [m]'); ax1.set_ylabel('zc [m]')
ax1.set_title(f'Sensitivity: xc−zc scan at Lx={Lx_fixed}, Lz={Lz_fixed} m (area={Lx_fixed*Lz_fixed:.4f} m²)')
plt.tight_layout(); plt.savefig('sensitivity_heatmap.png', dpi=150); plt.show()

# ============================================================
# PART A — Ultra-Dense NSGA-II Pareto Front
# ============================================================
class SurrogateProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=4, n_obj=2, n_ieq_constr=4,
            xl=lb, xu=ub
        )

    def _evaluate(self, x, out, *args, **kwargs):
        xc, zc, Lx, Lz = x[0], x[1], x[2], x[3]
        g1 = Lx/2 - xc
        g2 = xc + Lx/2 - 10.0
        g3 = Lz/2 - zc
        g4 = zc + Lz/2 - 3.0
        area = Lx * Lz
        outage = float(predict_outage(x.reshape(1, -1))[0])
        out["F"] = [area, outage]
        out["G"] = [g1, g2, g3, g4]

problem = SurrogateProblem()

algorithm = NSGA2(
    pop_size=500,
    n_offsprings=250,
    sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9, eta=15),
    mutation=PM(prob=0.25, eta=20),
    eliminate_duplicates=True
)

termination = get_termination("n_gen", 300)

print("="*60)
print(" NSGA-II with surrogate: pop=500, gen=300")
print(" ~150k evaluations on GPU (μs each)")
print("="*60)

res = minimize(problem, algorithm, termination, seed=42, verbose=True)

print(f"\nPareto front: {len(res.F)} points")
print(f"Area range:   [{res.F[:,0].min():.4f}, {res.F[:,0].max():.2f}] m²")
print(f"Outage range: [{res.F[:,1].min()*100:.2f}%, {res.F[:,1].max()*100:.2f}%]")

# ============================================================
# Pareto Front Plot
# ============================================================
F = res.F

# Non-dominated filter
is_dom = np.zeros(len(F), dtype=bool)
for i in range(len(F)):
    for j in range(len(F)):
        if i != j and F[j,0]<=F[i,0] and F[j,1]<=F[i,1] and (F[j,0]<F[i,0] or F[j,1]<F[i,1]):
            is_dom[i] = True; break
pareto_F = F[~is_dom]
sort_idx = np.argsort(pareto_F[:,1])
pareto_F = pareto_F[sort_idx]

feasible = pareto_F[:,1] <= 0.10
if feasible.any():
    best_idx = np.argmin(pareto_F[feasible, 0])
    best_area = pareto_F[feasible, 0][best_idx]
    best_out  = pareto_F[feasible, 1][best_idx]

fig2, ax2 = plt.subplots(figsize=(9, 6))
ax2.scatter(F[:,1]*100, F[:,0], c='steelblue', s=2, alpha=0.15, label='All evaluated')
ax2.plot(pareto_F[:,1]*100, pareto_F[:,0], 'o-', color='darkgreen',
         markersize=4, linewidth=1.5, label=f'Pareto-optimal ({len(pareto_F)} pts)')
ax2.axvline(x=10, color='red', linestyle='--', alpha=0.6, label='10% outage')

if feasible.any():
    ax2.plot(best_out*100, best_area, 'r*', markersize=16, markeredgewidth=2,
            label=f'Best feasible: {best_area:.4f} m² @ {best_out*100:.1f}%')

ax2.set_xlabel('Outage [%]'); ax2.set_ylabel('Area [m²]')
ax2.set_title('Surrogate NSGA-II: Ultra-Dense Pareto Front (pop=500, gen=300)')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 40); ax2.set_ylim(0, None)
plt.tight_layout(); plt.savefig('pareto_surrogate.png', dpi=150); plt.show()

# ============================================================
# Knee Solution
# ============================================================
if feasible.any():
    feasible_all = np.where(F[:,1] <= 0.10)[0]
    knee_idx = feasible_all[np.argmin(F[feasible_all, 0])]
    knee_x = res.X[knee_idx]
    print("\n" + "="*60)
    print("Surrogate Knee Solution (smallest area, outage <= 10%)")
    print("="*60)
    print(f"  xc={knee_x[0]:.3f}m  zc={knee_x[1]:.3f}m  Lx={knee_x[2]:.3f}m  Lz={knee_x[3]:.3f}m")
    print(f"  Area={F[knee_idx,0]:.4f} m²   Outage={F[knee_idx,1]*100:.2f}%")
    print("="*60)

print("\nDone. Files: sensitivity_heatmap.png, pareto_surrogate.png")